# Métodos numéricos aplicados a un brazo robótico de 2 GDL

Simularemos un brazo plano de dos grados de libertad y compararemos cuatro integradores numéricos:

- **RK1 o Euler explícito:** primer orden y paso fijo.
- **RK4:** cuarto orden y paso fijo.
- **RK23 (Bogacki–Shampine):** paso adaptativo, orden 3 con estimación de error de orden 2.
- **RK45 (Dormand–Prince):** paso adaptativo, orden 5 con estimación de error de orden 4.

Al final construiremos:

1. una gráfica interactiva de los ángulos articulares;
2. una tabla que compara los métodos;
3. una animación interactiva del brazo.

> **Cómo usarlo en Colab:** selecciona `Entorno de ejecución → Ejecutar todas`. No se requiere GPU.

## 1. Modelo dinámico

El vector de estado es

$$
\mathbf{x}=\begin{bmatrix}\theta_1&\theta_2&\dot\theta_1&\dot\theta_2\end{bmatrix}^{\mathsf T},
$$

donde $\theta_1$ y $\theta_2$ son los ángulos articulares. La dinámica se escribe como

$$
M(\boldsymbol\theta)\,\ddot{\boldsymbol\theta}
+C(\boldsymbol\theta,\dot{\boldsymbol\theta})
=\boldsymbol\tau,
$$

de modo que

$$
\ddot{\boldsymbol\theta}=M(\boldsymbol\theta)^{-1}
\left[\boldsymbol\tau-C(\boldsymbol\theta,\dot{\boldsymbol\theta})\right].
$$

En el código no calculamos explícitamente $M^{-1}$; utilizamos `np.linalg.solve`, que suele ser más estable y eficiente. Cada articulación se limita aproximadamente al intervalo $[-\pi/2,\pi/2]$ mediante un par de penalización:

$$
\tau_p(\theta)=
\begin{cases}
-k(\theta-\theta_{\max}), & \theta>\theta_{\max},\\
k(\theta_{\min}-\theta), & \theta<\theta_{\min},\\
0, & \text{en otro caso}.
\end{cases}
$$

Este límite es una **aproximación mediante un resorte virtual**, no una restricción rígida ideal.

## 2. Preparación de Google Colab

Colab normalmente ya incluye NumPy, SciPy y Plotly. La siguiente celda los instala o actualiza silenciosamente para que el notebook sea reproducible.

In [1]:
%pip install -q numpy scipy plotly pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from scipy.integrate import solve_ivp

# Este renderer hace que las figuras interactivas se muestren correctamente en Colab.
pio.renderers.default = "colab"

np.set_printoptions(precision=5, suppress=True)

## 3. Parámetros físicos y límites articulares

Las longitudes se expresan en metros, las masas en kilogramos, los ángulos en radianes y el tiempo en segundos. Puedes modificar estos valores y volver a ejecutar desde esta celda.

In [3]:
# Longitudes y masas de los eslabones
l1, l2 = 1.0, 1.0
m1, m2 = 1.0, 1.0

# Aceleración gravitacional
g = 9.81

# Rigidez del resorte virtual usado en los límites articulares
k_penalty = 100.0

# Límites de ambas articulaciones
theta_min = -np.pi / 2
theta_max =  np.pi / 2

print(f"Límites: [{np.degrees(theta_min):.0f}°, {np.degrees(theta_max):.0f}°]")

Límites: [-90°, 90°]


## 4. Par de penalización

Mientras el ángulo permanece dentro de sus límites, el par es cero. Cuando los rebasa, aparece un par restaurador proporcional a la penetración.

In [4]:
def penalty_torque(theta, theta_min, theta_max, k=100.0):
    """Calcula el par restaurador asociado a un límite articular blando."""
    return -k * max(0.0, theta - theta_max) + k * max(0.0, theta_min - theta)


# Pequeña comprobación
test_angles = np.radians([-120, -90, 0, 90, 120])
for angle in test_angles:
    torque = penalty_torque(angle, theta_min, theta_max, k_penalty)
    print(f"theta = {np.degrees(angle):6.1f}°  ->  tau = {torque:8.3f} N·m")

theta = -120.0°  ->  tau =   52.360 N·m
theta =  -90.0°  ->  tau =    0.000 N·m
theta =    0.0°  ->  tau =    0.000 N·m
theta =   90.0°  ->  tau =    0.000 N·m
theta =  120.0°  ->  tau =  -52.360 N·m


## 5. Función de dinámica

Los integradores necesitan una función con la forma `f(t, x)` que devuelva $\dot{\mathbf{x}}$. A partir de

$$
\mathbf{x}=\begin{bmatrix}\theta_1&\theta_2&\dot\theta_1&\dot\theta_2\end{bmatrix}^{\mathsf T},
$$

la función devuelve

$$
\dot{\mathbf{x}}=
\begin{bmatrix}\dot\theta_1&\dot\theta_2&\ddot\theta_1&\ddot\theta_2\end{bmatrix}^{\mathsf T}.
$$

In [5]:
def dynamics(t, state):
    """Modelo no lineal del brazo robótico plano de 2 GDL."""
    theta1, theta2, dtheta1, dtheta2 = state

    tau1 = penalty_torque(theta1, theta_min, theta_max, k_penalty)
    tau2 = penalty_torque(theta2, theta_min, theta_max, k_penalty)

    # Matriz de inercia M(theta)
    M11 = m1 * l1**2 / 3 + m2 * (
        l1**2 + l2**2 / 3 + l1 * l2 * np.cos(theta2)
    )
    M12 = m2 * (l2**2 / 3 + l1 * l2 / 2 * np.cos(theta2))
    M21 = M12
    M22 = m2 * l2**2 / 3
    M = np.array([[M11, M12], [M21, M22]])

    # Términos centrífugos/Coriolis y gravitacionales del modelo original
    C1 = (
        -m2 * l1 * l2 / 2 * np.sin(theta2) * dtheta2**2
        - (m1 * l1 / 2 + m2 * l1) * g * np.sin(theta1)
    )
    C2 = (
        m2 * l1 * l2 / 2 * np.sin(theta2) * dtheta1**2
        - m2 * l2 / 2 * g * np.sin(theta1 + theta2)
    )
    C = np.array([C1, C2])

    tau = np.array([tau1, tau2])
    ddtheta = np.linalg.solve(M, tau - C)

    return np.array([dtheta1, dtheta2, ddtheta[0], ddtheta[1]])


# Derivada inicial: [velocidades, aceleraciones]
initial_test_state = np.array([np.pi / 4, np.pi / 4, 0.0, 0.0])
print("dx/dt en el estado inicial:", dynamics(0.0, initial_test_state))

dx/dt en el estado inicial: [ 0.       0.       0.31047 14.07522]


## 6. Integradores de paso fijo

### Euler explícito (RK1)

$$
\mathbf{x}_{k+1}=\mathbf{x}_k+h f(t_k,\mathbf{x}_k).
$$

Es sencillo y rápido por paso, pero su error global es de orden $O(h)$. Puede requerir un paso muy pequeño.

### Runge–Kutta de cuarto orden (RK4)

$$
\begin{aligned}
k_1 &= f(t_k,x_k),\\
k_2 &= f(t_k+h/2,x_k+hk_1/2),\\
k_3 &= f(t_k+h/2,x_k+hk_2/2),\\
k_4 &= f(t_k+h,x_k+hk_3),\\
x_{k+1} &= x_k+\frac{h}{6}(k_1+2k_2+2k_3+k_4).
\end{aligned}
$$

Su error global es de orden $O(h^4)$.

In [6]:
def fixed_time_grid(t_span, dt):
    """Crea una malla que termina exactamente en tf."""
    t0, tf = t_span
    n_steps = int(np.ceil((tf - t0) / dt))
    return np.linspace(t0, tf, n_steps + 1)


def rk1_solver(f, t_span, y0, dt):
    """Método de Euler explícito (RK1) con paso fijo."""
    times = fixed_time_grid(t_span, dt)
    states = np.zeros((len(times), len(y0)), dtype=float)
    states[0] = y0

    for i in range(len(times) - 1):
        h = times[i + 1] - times[i]
        states[i + 1] = states[i] + h * np.asarray(f(times[i], states[i]))

    return times, states


def rk4_solver(f, t_span, y0, dt):
    """Método clásico de Runge–Kutta de cuarto orden con paso fijo."""
    times = fixed_time_grid(t_span, dt)
    states = np.zeros((len(times), len(y0)), dtype=float)
    states[0] = y0

    for i in range(len(times) - 1):
        t = times[i]
        y = states[i]
        h = times[i + 1] - times[i]

        k1 = np.asarray(f(t, y))
        k2 = np.asarray(f(t + h / 2, y + h * k1 / 2))
        k3 = np.asarray(f(t + h / 2, y + h * k2 / 2))
        k4 = np.asarray(f(t + h, y + h * k3))

        states[i + 1] = y + h * (k1 + 2*k2 + 2*k3 + k4) / 6

    return times, states

## 7. Configuración y ejecución

`RK23` y `RK45` ajustan internamente el tamaño de paso para satisfacer `rtol` y `atol`. `t_eval` solo indica los instantes en los que deseamos recibir la solución; no fuerza al integrador a utilizar esos pasos internos.

Para RK1 y RK4 sí elegimos un paso fijo `dt`. Un valor menor suele mejorar la aproximación, pero incrementa el tiempo de cálculo.

In [7]:
# Condición inicial
y0 = np.array([np.pi / 4, np.pi / 4, 0.0, 0.0])

# Intervalo de simulación y puntos usados para visualizar los métodos adaptativos
t_span = (0.0, 10.0)
t_eval = np.linspace(t_span[0], t_span[1], 201)

# Paso de los métodos explícitos de paso fijo
dt = 0.001

# Tolerancias de los métodos adaptativos
rtol = 1e-7
atol = 1e-9

In [8]:
results = {}
execution_times = {}

start = time.perf_counter()
sol_rk45 = solve_ivp(
    dynamics, t_span, y0, t_eval=t_eval, method="RK45", rtol=rtol, atol=atol
)
execution_times["RK45"] = time.perf_counter() - start
if not sol_rk45.success:
    raise RuntimeError(sol_rk45.message)
results["RK45"] = (sol_rk45.t, sol_rk45.y.T)

start = time.perf_counter()
sol_rk23 = solve_ivp(
    dynamics, t_span, y0, t_eval=t_eval, method="RK23", rtol=rtol, atol=atol
)
execution_times["RK23"] = time.perf_counter() - start
if not sol_rk23.success:
    raise RuntimeError(sol_rk23.message)
results["RK23"] = (sol_rk23.t, sol_rk23.y.T)

start = time.perf_counter()
t_rk1, y_rk1 = rk1_solver(dynamics, t_span, y0, dt)
execution_times["RK1 (Euler)"] = time.perf_counter() - start
results["RK1 (Euler)"] = (t_rk1, y_rk1)

start = time.perf_counter()
t_rk4, y_rk4 = rk4_solver(dynamics, t_span, y0, dt)
execution_times["RK4"] = time.perf_counter() - start
results["RK4"] = (t_rk4, y_rk4)

print("Simulaciones terminadas correctamente.")

Simulaciones terminadas correctamente.


## 8. Comparación de los ángulos

Usa la leyenda para ocultar o mostrar curvas. También puedes hacer zoom, desplazar la vista y guardar la figura desde la barra de herramientas de Plotly.

In [11]:
styles = {
    "RK45": {"theta1": "#009E73", "theta2": "#E69F00", "dash": "solid"},
    "RK23": {"theta1": "#009E73", "theta2": "#E69F00", "dash": "dash"},
    "RK1 (Euler)": {"theta1": "#0072B2", "theta2": "#D55E00", "dash": "dot"},
    "RK4": {"theta1": "#0072B2", "theta2": "#D55E00", "dash": "solid"},
}

angle_plot = go.Figure()

for method, (times, states) in results.items():
    style = styles[method]
    angle_plot.add_trace(go.Scatter(
        x=times, y=states[:, 0], mode="lines",
        name=f"θ₁ — {method}",
        line=dict(color=style["theta1"], dash=style["dash"]),
        hovertemplate="t=%{x:.3f} s<br>θ₁=%{y:.5f} rad<extra></extra>",
    ))
    angle_plot.add_trace(go.Scatter(
        x=times, y=states[:, 1], mode="lines",
        name=f"θ₂ — {method}",
        line=dict(color=style["theta2"], dash=style["dash"]),
        hovertemplate="t=%{x:.3f} s<br>θ₂=%{y:.5f} rad<extra></extra>",
    ))

angle_plot.update_layout(
    title="Comparación de los ángulos articulares",
    xaxis_title="Tiempo (s)",
    yaxis_title="Ángulo (rad)",
    template="plotly_white",
    height=620,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
angle_plot.show()

<!doctype html>

## 9. Comparación cuantitativa

Tomaremos RK45 con tolerancias estrictas como referencia práctica y calcularemos el error cuadrático medio (RMSE) de los ángulos. Esto sirve para comparar estas ejecuciones, pero **no constituye una solución analítica exacta**.

Las soluciones de paso fijo se interpolan en `t_eval` para que todas se comparen en los mismos instantes.

In [13]:
reference = sol_rk45.y.T
summary_rows = []

for method, (times, states) in results.items():
    theta1_on_eval = np.interp(t_eval, times, states[:, 0])
    theta2_on_eval = np.interp(t_eval, times, states[:, 1])

    rmse_theta1 = np.sqrt(np.mean((theta1_on_eval - reference[:, 0])**2))
    rmse_theta2 = np.sqrt(np.mean((theta2_on_eval - reference[:, 1])**2))

    if method == "RK45":
        evaluations_or_steps = sol_rk45.nfev
        measure = "evaluaciones de f"
    elif method == "RK23":
        evaluations_or_steps = sol_rk23.nfev
        measure = "evaluaciones de f"
    else:
        evaluations_or_steps = len(times) - 1
        measure = "pasos fijos"

    summary_rows.append({
        "Método": method,
        "RMSE θ₁ (rad)": rmse_theta1,
        "RMSE θ₂ (rad)": rmse_theta2,
        "Cantidad": evaluations_or_steps,
        "Medida": measure,
        "Tiempo (s)": execution_times[method],
    })

summary = pd.DataFrame(summary_rows)
summary_display = summary.copy()
summary_display["RMSE θ₁ (rad)"] = summary["RMSE θ₁ (rad)"].map("{:.3e}".format)
summary_display["RMSE θ₂ (rad)"] = summary["RMSE θ₂ (rad)"].map("{:.3e}".format)
summary_display["Tiempo (s)"] = summary["Tiempo (s)"].map("{:.4f}".format)
summary_display

,Método,RMSE θ₁ (rad),RMSE θ₂ (rad),Cantidad,Medida,Tiempo (s)
0,RK45,0.000e+00,0.000e+00,11180,evaluaciones de f,0.1679
1,RK23,3.089e-03,6.671e-03,42638,evaluaciones de f,0.9052
2,RK1 (Euler),1.494e+00,1.567e+00,10000,pasos fijos,0.1506
3,RK4,9.154e-03,2.026e-02,10000,pasos fijos,0.5785


> **Precaución al interpretar el tiempo:** puede variar entre ejecuciones y equipos. Además, una evaluación de `dynamics` no equivale a un paso completo; RK4, por ejemplo, utiliza cuatro evaluaciones por paso.

## 10. Cinemática directa y animación

Para dibujar el brazo usamos la solución de RK45. Las posiciones de las articulaciones son

$$
\begin{aligned}
x_1 &= l_1\cos\theta_1, & y_1 &= l_1\sin\theta_1,\\
x_2 &= x_1+l_2\cos(\theta_1+\theta_2), &
y_2 &= y_1+l_2\sin(\theta_1+\theta_2).
\end{aligned}
$$

Aquí $\theta_2$ se interpreta como un ángulo **relativo al primer eslabón**, por eso la orientación absoluta del segundo eslabón es $\theta_1+\theta_2$.

In [12]:
theta1_rk45 = sol_rk45.y[0]
theta2_rk45 = sol_rk45.y[1]

x1 = l1 * np.cos(theta1_rk45)
y1 = l1 * np.sin(theta1_rk45)
x2 = x1 + l2 * np.cos(theta1_rk45 + theta2_rk45)
y2 = y1 + l2 * np.sin(theta1_rk45 + theta2_rk45)

# Usamos como máximo 101 cuadros para mantener el notebook ligero.
frame_indices = np.unique(np.linspace(0, len(t_eval) - 1, min(101, len(t_eval))).astype(int))

frames = [
    go.Frame(
        name=str(i),
        data=[go.Scatter(
            x=[0, x1[i], x2[i]],
            y=[0, y1[i], y2[i]],
            mode="lines+markers",
            line=dict(color="#0072B2", width=5),
            marker=dict(size=[10, 12, 12], color=["black", "#D55E00", "#D55E00"]),
        )],
        layout=go.Layout(title_text=f"Brazo de 2 GDL — RK45 — t = {t_eval[i]:.2f} s"),
    )
    for i in frame_indices
]

arm_plot = go.Figure(
    data=[go.Scatter(
        x=[0, x1[0], x2[0]],
        y=[0, y1[0], y2[0]],
        mode="lines+markers",
        line=dict(color="#0072B2", width=5),
        marker=dict(size=[10, 12, 12], color=["black", "#D55E00", "#D55E00"]),
    )],
    frames=frames,
)

arm_plot.update_layout(
    title="Brazo de 2 GDL — RK45 — t = 0.00 s",
    template="plotly_white",
    width=700,
    height=650,
    showlegend=False,
    xaxis=dict(range=[-(l1+l2)*1.1, (l1+l2)*1.1], title="x (m)", scaleanchor="y", scaleratio=1),
    yaxis=dict(range=[-(l1+l2)*1.1, (l1+l2)*1.1], title="y (m)"),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0.5,
        xanchor="center",
        buttons=[
            dict(label="▶ Reproducir", method="animate",
                 args=[None, {"frame": {"duration": 60, "redraw": True},
                              "fromcurrent": True, "transition": {"duration": 0}}]),
            dict(label="⏸ Pausar", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False},
                                "mode": "immediate", "transition": {"duration": 0}}]),
        ],
    )],
)

arm_plot.show()

<!doctype html>

## 11. Experimentos sugeridos

Prueba una modificación a la vez y vuelve a ejecutar las celdas desde la configuración:

1. Cambia `dt` a `0.01`, `0.005` y `0.0005`. Observa cómo cambia RK1 respecto de RK45.
2. Cambia las tolerancias de RK23 y RK45, por ejemplo a `rtol=1e-4` y `atol=1e-6`.
3. Incrementa `k_penalty`. Un resorte virtual más rígido puede volver el problema numéricamente más exigente.
4. Cambia la condición inicial para comenzar fuera de los límites, por ejemplo `y0=[2.0, 0.5, 0, 0]`.
5. Registra `sol_rk45.nfev` y `sol_rk23.nfev` para estudiar el costo de tolerancias más estrictas.

### Idea principal

No existe un integrador universalmente mejor. La selección depende de la precisión requerida, el costo computacional y las características dinámicas del sistema. Euler es útil para entender la idea básica; RK4 ofrece buena precisión con paso fijo; RK23 y RK45 controlan el error y ajustan automáticamente su paso interno.